# two ways to stream Anthropic API responses — the low-level raw event loop vs the high-level SDK helper.

In [5]:
from dotenv import load_dotenv
load_dotenv()

import httpx
from anthropic import Anthropic

client = Anthropic(http_client=httpx.Client(verify=False))

model = "claude-sonnet-4-6"


In [6]:
def add_user_message(messages, content):
    messages.append({
        "role": "user",
        "content": content
    })

def add_assistant_message(messages, content):
    messages.append({
        "role": "assistant",
        "content": content
    })

def chat_log (messages,system=None):
    # we do if condition to avoid sending system as None to the API, which will cause err
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stream": True
    }

    if system: 
        params["system"] = system

    streaming = client.messages.create(**params)

    for event in streaming:
        print (event)

def chat_stream_internal_print(messages, system= None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }
    if system: 
        params["system"] = system 

    with client.messages.stream(**params) as stream:
        for txt in stream.text_stream:
            print (txt, end = "") 

def chat_stream(messages, system= None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }
    if system: 
        params["system"] = system 

    stream_msg = client.messages.stream(**params)


    return stream_msg

In [8]:
messages = []
system = '''
you are expert in geography
    '''
add_user_message (messages, "tell me about singapore in 50 words")

stream1 = chat_stream(messages=messages , system = system)

with stream1 as stream:
    for txt in stream.text_stream:
        pass

stream.get_final_message().content[0].text
# chat_stream_internal_print(messages= messages, system=system)





"**Singapore** is a small island city-state in Southeast Asia. Despite its tiny size (~728 km²), it's one of the world's wealthiest nations. Known for its **multicultural population** (Chinese, Malay, Indian), modern skyline, efficient governance, bustling port, and landmarks like Marina Bay Sands and Gardens by the Bay."